# Redrob Ranker — Sandbox

Runs the ranking system end-to-end on a small sample (50 candidates from `sample_candidates.json`) and produces a ranked CSV.

**How to use:** `Runtime → Run all`. A CPU runtime is fine (no GPU needed). Total time is a few minutes (mostly one-time model downloads).

This demonstrates small-sample reproducibility per submission_spec Section 10.5. The full 100K run is reproduced separately from the GitHub repo.

In [ ]:
# 1. Clone the repo and install dependencies
!git clone --depth 1 https://github.com/parthrajsinghbhati/redrob.git
%cd redrob
!pip install -q -r requirements.txt

In [ ]:
# 2. Build a small JSONL sample from the bundled 50-candidate file
import json

data = json.load(open("India_runs_data_and_ai_challenge/sample_candidates.json"))
with open("sample.jsonl", "w") as f:
    for c in data:
        f.write(json.dumps(c) + "\n")
print(f"Wrote {len(data)} candidates to sample.jsonl")

In [ ]:
# 3. Pre-compute dense embeddings for the sample (CPU, ~seconds for 50 candidates)
!python precompute_embeddings.py --candidates sample.jsonl --out-dir data

In [ ]:
# 4. Run the ranking pipeline end-to-end and show the ranked output.
# We pass --no-rerank here so the sandbox stays comfortably inside the
# 5-min CPU budget (the cross-encoder's ~1.1 GB one-time download is the only
# thing that could blow it, and re-ranking a 50-row sample changes nothing).
# The FULL submission pipeline (dense + BM25 + rule + cross-encoder re-rank)
# is the documented command in the README and is what produced the real CSV:
#   python rank.py --candidates candidates.jsonl --out submission.csv
!python rank.py --candidates sample.jsonl --out sample_output.csv --no-rerank

import pandas as pd
df = pd.read_csv("sample_output.csv")
print(f"Ranked {len(df)} candidates")
df